# 07 · Sensibilidad y fallas de producción (parte de JME)

Notebook de demostración de la parte 07 (`code/evaluate.py`, sección *Parte 07*). Guía:
`docs/instrucciones/07_sensibilidad_produccion.md`. El simulador supone que el resto del sistema es gratuito, instantáneo
y perfecto; aquí se mide cuánto cambia el resultado al relajar cada supuesto, **uno a la vez**, con el agente congelado
de la parte 05 (el modelo elegido en validación de cada fold, `artifacts/run_metadata.json`) y, como referencia, las
líneas base *siempre largo* y *aleatorio*. Necesita los modelos de `artifacts/models/` (los escribe `python main.py`).
La ablación y el segundo algoritmo (pregunta 7) son de GC.

In [1]:
import json
import sys
sys.path.insert(0, "..")
sys.modules.pop("code", None)  # IPython ya importó el módulo `code` estándar (ver docs/decisiones.md)

import pandas as pd

from code import data, env, evaluate, train
from main import ARTIFACTS, BTC_CSV, FOLDS, MODELS, SEED

ohlcv, _ = data.load_ohlcv(BTC_CSV)
features = data.compute_features(ohlcv)
pd.set_option("display.width", 200)

chosen = json.loads((ARTIFACTS / "run_metadata.json").read_text())["chosen"]
agent = {int(k): train.as_policy(train.DQN.load(train.model_path(MODELS, int(k), c["config_id"], c["seed"], c["timesteps"]),
                                                device="cpu"))
         for k, c in chosen.items()}

## 1. Los escenarios

Cada uno cambia un solo parámetro del entorno respecto a la base (stop primero, sin retraso, 0.125 % por lado):

In [2]:
fold = data.build_fold_dataset(ohlcv, features, 3, FOLDS[3])
latency = evaluate.latency_cost(fold.raw_features["train"])
pd.Series(evaluate.sensitivity_scenarios(env.COST_PER_SIDE, latency), name="parámetros del entorno").to_frame()

,parámetros del entorno
base,{}
tp_first,{'intrabar': 'tp_first'}
retraso_1_barra,{'obs_lag': 1}
latencia_2s,{'cost': 0.0013894538097869143}
costo_+5pb,{'cost': 0.00175}
costo_+10pb,{'cost': 0.0022500000000000003}
costo_+25pb,{'cost': 0.00375}


## 2. Latencia de 2 s como costo

El agente decide al cierre de t y se llena a la apertura de t + 1, que en Binance coincide con el cierre de t (mediana de
|apertura / cierre previo| ≈ 0 entre barras contiguas, parte 01). Si la inferencia tarda 2 s, se pierde el movimiento de
esos 2 s: con rendimientos sin deriva, su tamaño típico es σ̂·√(2/3600). Se agrega como costo extra por lado.

In [3]:
pd.DataFrame({k: {"σ̂ horaria mediana (train)": data.split_fold(features, FOLDS[k])["train"]["belief_vol"].median(),
                  "costo por lado en 2 s (pb)": 1e4 * evaluate.latency_cost(data.split_fold(features, FOLDS[k])["train"])}
              for k in FOLDS}).T.round(5)

,σ̂ horaria mediana (train),costo por lado en 2 s (pb)
1,0.00663,1.56256
2,0.00638,1.50293
3,0.00592,1.39454
4,0.00571,1.34641


## 3. Una prueba completa con el agente (fold 2, 2024)

`same_trades_share` es la proporción de operaciones de la base que el escenario repite (misma hora de decisión y mismo
lado). Con 1, el agente decidió lo mismo y el cambio es directo; por debajo de 1, tomó otras decisiones.

In [4]:
fold = data.build_fold_dataset(ohlcv, features, 2, FOLDS[2])
table = evaluate.run_sensitivity(fold, lambda: agent[2], FOLDS[2]["test"], SEED + 2, env.COST_PER_SIDE)
table[["total_return", "sharpe", "max_drawdown", "n_trades", "cost_per_side", "n_ambiguous", "same_trades_share",
       "delta_total_return", "delta_sharpe"]].round(4)

,total_return,sharpe,max_drawdown,n_trades,cost_per_side,n_ambiguous,same_trades_share,delta_total_return,delta_sharpe
scenario,,,,,,,,,
base,-0.2544,-1.0378,-0.3743,142.0,0.0012,0.0,1.0000,0.0000,0.0000
tp_first,-0.2544,-1.0378,-0.3743,142.0,0.0012,0.0,1.0000,0.0000,0.0000
retraso_1_barra,-0.2326,-0.7284,-0.3385,138.0,0.0012,0.0,0.0915,0.0218,0.3094
latencia_2s,0.0119,0.1742,-0.2095,138.0,0.0014,0.0,0.7817,0.2662,1.2120
costo_+5pb,-0.3089,-1.3097,-0.4330,136.0,0.0018,0.0,0.7535,-0.0546,-0.2719
costo_+10pb,-0.4483,-2.2032,-0.5188,135.0,0.0023,0.0,0.6972,-0.1939,-1.1654
costo_+25pb,-0.6643,-3.8925,-0.6796,129.0,0.0038,0.0,0.5211,-0.4099,-2.8547


## 4. Los cuatro folds (`artifacts/sensitivity.csv`)

Media y desviación entre folds del cambio contra la base, y proporción de operaciones repetidas por fold.

- **Intrabarra:** el agente no tiene ninguna operación con stop y take-profit en la misma barra en las cuatro pruebas
  (379 operaciones); en todo el artefacto hay una sola, del aleatorio. El orden dentro de la barra no mueve el resultado.
- **Costo extra:** +5/+10/+25 pb por lado quitan en promedio 4.9/11.6/26.3 puntos de rendimiento total al agente.
- **Retraso de una barra:** el agente repite solo 6–14 % de sus operaciones: sus decisiones dependen de la última barra.
- **Latencia de 2 s:** casi no cambia los folds 1, 3 y 4, pero en el fold 2 "gana" 26.6 puntos (sección 5).

In [5]:
sens = pd.read_csv("../artifacts/sensitivity.csv", index_col=[0, 1, 2])
display(sens.loc[["media", "desv"], ["delta_total_return", "delta_sharpe"]].unstack(0).round(4))
sens.drop(["media", "desv"], level="fold")["same_trades_share"].unstack("scenario").round(3)

delta_total_return         delta_sharpe        
fold                                       media    desv        media    desv
strategy      scenario                                                       
agente        base                        0.0000  0.0000       0.0000  0.0000
              costo_+10pb                -0.1162  0.0832      -0.8076  0.2894
              costo_+25pb                -0.2631  0.1631      -2.1035  0.6324
              costo_+5pb                 -0.0489  0.0351      -0.3338  0.0970
              latencia_2s                 0.0648  0.1344       0.2669  0.6326
              retraso_1_barra            -0.0572  0.0678      -0.5339  0.9909
              tp_first                    0.0000  0.0000       0.0000  0.0000
aleatorio     base                        0.0000  0.0000       0.0000  0.0000
              costo_+10pb                -0.1515  0.0573      -2.2835  0.4020
              costo_+25pb                -0.2532  0.1504      -5.6105  0.9432
              costo_+5pb                 -0.0895  0.0261      -1.1462  0.2045
              latencia_2s                -0.0294  0.0059      -0.3355  0.0791
              retraso_1_barra            -0.1125  0.1367      -1.1656  1.4844
              tp_first                    0.0011  0.0022       0.0138  0.0276
siempre_largo base                        0.0000  0.0000       0.0000  0.0000
              costo_+10pb                -0.2560  0.1482      -2.2312  0.4104
              costo_+25pb                -0.3875  0.1923      -5.5058  1.0573
              costo_+5pb                 -0.1577  0.0974      -1.1184  0.2024
              latencia_2s                -0.0553  0.0390      -0.3273  0.0766
              retraso_1_barra            -0.0139  0.0618      -0.0432  0.2806
              tp_first                    0.0000  0.0000       0.0000  0.0000

scenario            base  costo_+10pb  costo_+25pb  costo_+5pb  latencia_2s  retraso_1_barra  tp_first
fold strategy                                                                                         
1    agente          1.0        0.970        0.697       0.970        1.000            0.121       1.0
     aleatorio       1.0        1.000        1.000       1.000        1.000            0.136       1.0
     siempre_largo   1.0        1.000        1.000       1.000        1.000            0.686       1.0
2    agente          1.0        0.697        0.521       0.754        0.782            0.092       1.0
     aleatorio       1.0        1.000        1.000       1.000        1.000            0.202       1.0
     siempre_largo   1.0        1.000        1.000       1.000        1.000            0.717       1.0
3    agente          1.0        0.839        0.667       0.946        0.952            0.140       1.0
     aleatorio       1.0        1.000        1.000       1.000        1.000            0.230       1.0
     siempre_largo   1.0        1.000        1.000       1.000        1.000            0.594       1.0
4    agente          1.0        1.000        1.000       1.000        1.000            0.056       1.0
     aleatorio       1.0        1.000        1.000       1.000        1.000            0.357       1.0
     siempre_largo   1.0        1.000        1.000       1.000        1.000            0.673       1.0

## 5. Fragilidad: un costo un poco mayor cambia las decisiones

En el fold 2 la latencia sube el costo de 0.125 % a ≈ 0.140 % por lado. Si el agente repitiera sus 142 operaciones, el
efecto directo sería de ≈ −1.5 pb × 2 lados × 142 ≈ −4 puntos. En cambio, el rendimiento sube 26.6 puntos: el estado
incluye variables de la cuenta (`equity_drawdown`, `last_trade_return`, `hours_flat`), así que un capital ligeramente
distinto lleva a otras decisiones, y esas decisiones distintas pesan mucho más que el costo. La "mejora" no es una
ventaja de pagar más: muestra que el resultado de prueba del agente cambia mucho con perturbaciones pequeñas y que
la diferencia contra no operar está dentro de ese ruido.

In [6]:
logs = {}
for name, cost in {"base": env.COST_PER_SIDE,
                   "latencia_2s": env.COST_PER_SIDE + evaluate.latency_cost(fold.raw_features["train"])}.items():
    trading_env = env.make_env(fold, "test", lambda outcome: 0.0, mode="eval", cost=cost)
    logs[name] = evaluate.run_policy(trading_env, agent[2], SEED + 2)[1]
pd.Series({"operaciones (base)": len(logs["base"]), "operaciones (latencia)": len(logs["latencia_2s"]),
           "repetidas": evaluate.same_trades_share(logs["base"], logs["latencia_2s"]),
           "costo por lado extra (pb)": 1e4 * evaluate.latency_cost(fold.raw_features["train"]),
           "P&L neto base": logs["base"]["net_pnl"].sum(),
           "P&L neto latencia": logs["latencia_2s"]["net_pnl"].sum()}).round(4)

operaciones (base)           142.0000
operaciones (latencia)       138.0000
repetidas                      0.7817
costo por lado extra (pb)      1.5029
P&L neto base                 -0.2544
P&L neto latencia              0.0119
dtype: float64